# Stage1 10:32 首小时签名量：0bp净恢复优势信息检验

单一坐标、旧21项匹配对照、2018–2020训练/2021–2022复用评价，固定费用和统计门见 plan.json。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder
from src.workflow.intraday_recovery_direct_allocation import FEATURE_COLUMNS

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/stage1_signed_volume_0bp_action_information_20261023'
plan=json.loads((card/'plan.json').read_text());source=plan['sources']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists()
assert plan['budget']=={'new_external_requests':0,'source_Recorder_object_loads':2,'DatasetH_builds':1,'model_fit_attempts_max':2,'bootstrap_attempts_max':1,'new_writable_Qlib_Recorders_max':1,'native_accounts':0}
for prefix,artifact in [('action_feature','analysis/daily'),('label','analysis/labels')]:
    c=root/source[prefix+'_card'];s=json.loads((c/'summary.json').read_text());m=json.loads((c/'verification_manifest.json').read_text());b=json.loads((c/'nas_binding.json').read_text())
    assert sha(c/'summary.json')==source[prefix+'_summary_sha256'] and sha(c/'verification_manifest.json')==source[prefix+'_manifest_sha256']
    assert s['recorder_id']==source[prefix+'_recorder_id'] and s['nas_evidence']['sha256']==b['evidence']['sha256']==source[prefix+'_NAS_sha256']
    restore=json.loads((c/'nas_scoped_restore.json').read_text());assert restore['status']=='verified' and restore['archive_sha256']==source[prefix+'_NAS_sha256']
    matches=[c/e['path'] for e in m['entries'] if e['path'].endswith('/'+artifact)]
    assert len(matches)==1 and sha(matches[0])==source[prefix+'_artifact_sha256']
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
assert len(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day'))==plan['cohorts']['calendar_rows']
def load(prefix,artifact):
    c=root/source[prefix+'_card']
    with R.uri_context(str(c/'qlib_recorder')):
        return R.get_recorder(experiment_name=c.name,recorder_id=source[prefix+'_recorder_id']).load_object(artifact)
feature=load('action_feature','analysis/daily')
labels=load('label','analysis/labels')
old_names=list(FEATURE_COLUMNS);assert len(old_names)==21 and len(set(old_names))==21
assert len(labels)==plan['cohorts']['calendar_rows'] and labels.trade_date.is_unique
coord=feature[['trade_date','pressure','status','assumed_available_at']].rename(columns={'pressure':'SOXX_first_hour_signed_volume_pressure','status':'source_status'})
joined=labels.merge(coord,on='trade_date',validate='one_to_one',sort=False)
assert len(joined)==len(labels) and joined.trade_date.equals(labels.trade_date)
assert joined.loc[joined.eligible,'source_status'].eq('known').all()
assert joined.loc[joined.eligible,'SOXX_first_hour_signed_volume_pressure'].notna().all()
decision_local=pd.to_datetime(joined.decision_at).dt.tz_convert('America/New_York').dt.tz_localize(None)
assert (pd.to_datetime(joined.assumed_available_at)<=decision_local).all()
joined['feature_available_before_decision']=pd.to_datetime(joined.assumed_available_at)<=decision_local
joined['label_mature']=joined.net_advantage.notna()&pd.to_datetime(joined.label_available_at).notna()
assert (pd.to_datetime(joined.loc[joined.label_mature,'label_available_at'])>pd.to_datetime(joined.loc[joined.label_mature,'decision_at'])).all()
cuttrain=pd.Timestamp(plan['cohorts']['train_label_mature_before']);cuteval=pd.Timestamp(plan['cohorts']['evaluation_label_mature_before'])
train_calendar=joined.trade_date.between(*plan['cohorts']['train_trade_dates'])
eval_calendar=joined.trade_date.between(*plan['cohorts']['evaluation_trade_dates'])
train_mask=train_calendar&joined.eligible&joined.label_mature&pd.to_datetime(joined.label_available_at).lt(cuttrain)
eval_mask=eval_calendar&joined.eligible&joined.label_mature&pd.to_datetime(joined.label_available_at).lt(cuteval)
assert train_calendar.sum()+eval_calendar.sum()==len(joined)
assert int(train_mask.sum())>=plan['cohorts']['minimum_mature_train_eligible'] and int(eval_mask.sum())>=plan['cohorts']['minimum_mature_eval_eligible']
by_eval_year={int(y):int((eval_mask&joined.trade_date.dt.year.eq(y)).sum()) for y in plan['cohorts']['evaluation_complete_years']}
assert all(x>=plan['cohorts']['minimum_each_eval_year'] for x in by_eval_year.values())
assert np.isfinite(joined.loc[train_mask|eval_mask,'SOXX_first_hour_signed_volume_pressure'].to_numpy(float)).all()
static_features=joined[old_names+['SOXX_first_hour_signed_volume_pressure']].copy()
meta=pd.DataFrame({'eligible':joined.eligible.to_numpy(bool),'label_available_at':joined.label_available_at,'train_scored':train_mask.to_numpy(bool),'eval_scored':eval_mask.to_numpy(bool),'source_known':joined.source_status.eq('known').to_numpy(bool)})
static=pd.concat({'feature':static_features,'label':joined[['net_advantage']],'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([pd.DatetimeIndex(joined.trade_date),['SOXL_SOXX_CASH']*len(joined)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_trade_dates']),'eval':tuple(plan['cohorts']['evaluation_trade_dates'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('eval',col_set=['feature','label','meta'])
train=tr.loc[tr[('meta','train_scored')].astype(bool)];held=he.loc[he[('meta','eval_scored')].astype(bool)]
assert len(train)==int(train_mask.sum()) and len(held)==int(eval_mask.sum())
assert (train[('meta','eligible')].astype(bool).all() and held[('meta','eligible')].astype(bool).all())
ytr=train[('label','net_advantage')].to_numpy(float);yhe=held[('label','net_advantage')].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
preflight={'status':'fixed_0bp_action_value_DatasetH_preflight_passed','plan_sha256':sha(card/'plan.json'),'calendar_rows':len(joined),'train_calendar_rows':int(train_calendar.sum()),'eval_calendar_rows':int(eval_calendar.sum()),'train_mature_eligible':len(train),'eval_mature_eligible':len(held),'eval_annual_support':by_eval_year,'unknown_source_days':int(joined.source_status.ne('known').sum()),'eligible_unknown_source_days':int((joined.eligible&joined.source_status.ne('known')).sum()),'new_model_fits':0,'native_accounts':0}
atomic_json(preflight,card/'preflight_result.json')
state={'status':'fit_attempting','plan_sha256':sha(card/'plan.json'),'fit_attempts':[],'fits_completed':[],'bootstrap_attempts':0,'native_accounts':0}
atomic_json(state,card/'run_state.json')
inputs={'baseline':(train['feature'][old_names].to_numpy(float),held['feature'][old_names].to_numpy(float)),'candidate':(train['feature'][old_names+['SOXX_first_hour_signed_volume_pressure']].to_numpy(float),held['feature'][old_names+['SOXX_first_hour_signed_volume_pressure']].to_numpy(float))}
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));atomic_json(state,card/'run_state.json')
    dataset.config(dump_all=True,recursive=True)
    recorder.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset,source_recorder_ids={k:source[k+'_recorder_id'] for k in ('action_feature','label')})
    for family in ('baseline','candidate'):
        assert len(state['fit_attempts'])<2 and family not in state['fit_attempts']
        state['fit_attempts'].append(family);atomic_json(state,card/'run_state.json')
        model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
        model.fit(inputs[family][0],ytr)
        prediction=model.predict(inputs[family][1])
        assert np.isfinite(prediction).all()
        models[family]=model;predictions[family]=prediction
        recorder.save_objects(artifact_path='models',**{family:model})
        recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'trade_date':pd.DatetimeIndex(held.index.get_level_values('datetime')),'prediction':prediction,'target':yhe})})
        state['fits_completed'].append(family);atomic_json(state,card/'run_state.json')
        recorder.log_metrics(completed_fits=len(state['fits_completed']))
    dates=pd.DatetimeIndex(held.index.get_level_values('datetime'))
    scored=pd.DataFrame({'trade_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
    scored['paired_mse_improvement']=(scored.target-scored.baseline_prediction)**2-(scored.target-scored.candidate_prediction)**2
    full=pd.DataFrame({'trade_date':joined.loc[eval_calendar,'trade_date'].to_numpy(),'paired_mse_improvement':np.nan})
    full=full.merge(scored,on='trade_date',how='left',suffixes=('_calendar',''),validate='one_to_one')
    full['paired_mse_improvement']=full['paired_mse_improvement'].fillna(full.pop('paired_mse_improvement_calendar'))
    assert len(full)==int(eval_calendar.sum()) and int(full.paired_mse_improvement.notna().sum())==len(held)
    annual=[{'year':int(year),'calendar_rows':len(g),'scored_rows':int(g.paired_mse_improvement.notna().sum()),'mean_paired_mse_improvement':float(g.paired_mse_improvement.mean())} for year,g in full.groupby(full.trade_date.dt.year,sort=True)]
    assert [x['year'] for x in annual]==plan['cohorts']['evaluation_complete_years']
    boot=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempts']=1;atomic_json(state,card/'run_state.json')
    interval,indices,replicates=full_calendar_bootstrap(full.paired_mse_improvement.to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws'],seed=boot['seed'],alpha=boot['one_sided_alpha'])
    assert interval['valid_replicates']>=9990
    coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
    metrics={'baseline_eval_mse':float(np.mean((scored.target-scored.baseline_prediction)**2)),'candidate_eval_mse':float(np.mean((scored.target-scored.candidate_prediction)**2)),'paired_mse_improvement':float(scored.paired_mse_improvement.mean()),'annual':annual,'added_standardized_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
    gates={'coefficient_positive':coefficient>0,'full_eval_mean_positive':metrics['paired_mse_improvement']>0,'each_eval_year_mean_positive':all(x['mean_paired_mse_improvement']>0 for x in annual),'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
    passed=bool(all(gates.values()))
    recorder.save_objects(artifact_path='analysis',scored=scored,full_eval_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices)
    recorder.log_metrics(paired_mse_improvement=metrics['paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
    recorder_id,experiment_id=recorder.id,str(recorder.experiment_id)
summary={'experiment':plan['experiment'],'decision':'conditional_development_signed_volume_action_information_pass_needs_full_Stage1_0bp_account' if passed else 'not_promoted_fixed_0bp_signed_volume_recovery_amount_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':preflight,'information_gate_pass':passed,'metrics':metrics,'gates':gates,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'evaluated_held_2023_2026':False,'historical_independence':False,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['2018-2022 OOF Stage1-architecture local action labels are reused development history and not independent forward evidence.','The training reference omits actual broker-fill and corporate-action parity; no 2023+ complete funded account was run.','The first-hour source 10:31 availability is a completed-bar research assumption, not certified historical first receipt.']}
atomic_json(summary,card/'summary.json')
binding=json.loads((root/source['label_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
state.update(status='completed',recorder_id=recorder_id,experiment_id=experiment_id)
atomic_json(state,card/'run_state.json')
(card/'conclusion.md').write_text(
    '# Stage1 10:32 首小时签名量：0bp恢复净优势信息检验\n\n'
    f'冻结OOF训练参考的1061个日历日，训练截止2020年末的成熟合格标签{len(train)}个，2021/2022复用评价成熟合格标签{len(held)}个；旧21项输入与追加1项在相同日期、同一0bp额外滑点且仍扣Tiered费的beta0.6净恢复/HOLD局部金额标签上各拟合一次Ridge。0新账户。\n\n'
    f'评价MSE：旧输入{metrics["baseline_eval_mse"]:.9g}，新输入{metrics["candidate_eval_mse"]:.9g}；配对改善{metrics["paired_mse_improvement"]:+.9g}。2021/2022年度改善分别{annual[0]["mean_paired_mse_improvement"]:+.9g}/{annual[1]["mean_paired_mse_improvement"]:+.9g}，候选标准化坐标系数{coefficient:+.9g}；固定63日区块、10000次、单侧家族校正下界{metrics["bootstrap_lower"]:+.9g}。全部503个评价日保留在Recorder，其中未合格/未成熟保持未知。\n\n'
    f'决定`{summary["decision"]}`，门槛{gates}。局部恢复优势不是完整策略财富；没有2023+原Stage1完整0bp候选账户，更没有2/5bp及独立前向。源首次接收时钟和训练参考公司行动仍有明确限制。不因局部正年度或系数重选窗口/方向。Recorder `{recorder_id}`，NAS SHA `{remote}`。\n'
)
print(summary['decision'],preflight['train_mature_eligible'],preflight['eval_mature_eligible'],metrics,gates,recorder_id)


not_promoted_fixed_0bp_signed_volume_recovery_amount_information_failed 173 265 {'baseline_eval_mse': 8.382540986858794e-05, 'candidate_eval_mse': 8.411174685358631e-05, 'paired_mse_improvement': -2.863369849983748e-07, 'annual': [{'year': 2021, 'calendar_rows': 252, 'scored_rows': 27, 'mean_paired_mse_improvement': -2.221964716713811e-06}, {'year': 2022, 'calendar_rows': 251, 'scored_rows': 238, 'mean_paired_mse_improvement': -6.674896501385049e-08}], 'added_standardized_coefficient': -0.0007040436718007645, 'bootstrap_lower': -3.282670480600959e-06, 'bootstrap_upper': 9.553070048030327e-07, 'bootstrap': {'paired_mean_loss_improvement': -2.8633698499837473e-07, 'lower_bound': -3.282670480600959e-06, 'upper_bound': 9.553070048030327e-07, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 503, 'scored_rows': 265}} {'coefficient_positive': False, 'full_eval_mean_positive': False, 'each_eval_year_mean_positive': False, 'bootstrap_lower_positive': False} 390716634bac4fcab

## 中文结论

同一0bp Tiered beta0.6净恢复/HOLD金额标签下，旧21项基线MSE为8.38254e-5，加首小时签名量后为8.41117e-5；配对改善−2.86337e-7，2021/2022均为负，新增系数−0.000704，校正区块下界−3.28267e-6。四门全败，固定信息版本`not_promoted`，不运行2023+完整账户。503个评价日、265个成熟合格样本、两模型和10,000区块复制保存在Qlib Recorder，NAS 17工件限定恢复核对通过。局部信息不等于SOXX费用后稳定超额。